# Random-features mass model -- NUTS (numpyro) instead of `tempest`

This notebook is the `numpyro`-NUTS analogue of
`examples/random_features_blocks.ipynb`. It tests
`CHIMERA.population.mass.paired.random_features_density` (the fixed
random-features / random-Fourier-features paired mass model) with its 4
`feature_scales` (the mixture of lengthscales spanning broad-body to
narrow-peak structure) treated as **sampled hyperparameters** with **narrow
priors** around the defaults, alongside `w_out`. Section 1 mirrors
`pt-nuts/examples/chimera_pt_nuts.py`'s numpyro-model structure, but runs a
plain, single-temperature `numpyro.infer.MCMC(NUTS(model))` instead of
parallel-tempered NUTS or `tempest`'s SMC sampler -- i.e. exactly the
"plain `numpyro.infer.MCMC(NUTS(model))` run" that `pt_nuts.sampler`'s own
docstring uses as its sanity-check baseline.

**What changes relative to `random_features_blocks.ipynb`, and why:**

- **Priors are `numpyro.distributions`, not `scipy.stats`.** `tempest` (an
  SMC/nested-style sampler) needs a `prior_transform(u)` mapping the unit
  cube to parameter space, hence scipy's `.ppf`. `numpyro`'s NUTS instead
  needs a *generative model*: a Python function that calls `numpyro.sample`
  once per random variable, so priors are declared as `numpyro.distributions`
  objects and sampled inside that function.
- **`w_out` is one vector-valued sample site, not 64 scalar ones.** Under
  `tempest`, every hyperparameter had to be a scalar entry in one flat
  `params_keys` vector (`w_out0` ... `w_out63`), because `prior_transform`
  and `log_likelihood` only ever see/return flat arrays. `numpyro` samples
  directly into a pytree, so all `n_features` output weights share one
  `Normal(0, w_out_prior_scale)` site via `.expand([n_features]).to_event(1)`
  -- exactly the pattern `chimera_pt_nuts.py` already uses for its NN's
  `Ws`/`bs`. The 4 `feature_scales` still get one `numpyro.sample` call each
  (`scale1`...`scale4`), since each block has different prior bounds.
- **No importance weights.** `tempest.Sampler.posterior(resample=True)`
  returns weighted samples that need resampling to be treated as i.i.d.
  posterior draws. NUTS produces (approximately) i.i.d. posterior draws
  directly, so there is no `weights` array anywhere below, and
  `arviz.from_numpyro` builds the `InferenceData` straight from
  `mcmc.get_samples(group_by_chain=True)` (no `xarray.Dataset` bookkeeping by
  hand).
- **Real NUTS diagnostics are available**: divergences, `r_hat`, and
  effective sample size per parameter, none of which `tempest`'s SMC output
  gives you. Section 1 prints these after the run.

**Section 0** is fully local (no cluster data, no `tempest`/`numpyro`-cluster
dependency beyond the pip package itself) and sanity-checks the plumbing:
prior-predictive draws of different lengthscale "blocks", capacity
diagnostics, and a jit/vmap check that sampling `feature_scales` doesn't
trigger per-particle recompilation -- unchanged from the `tempest` notebook,
just resampled with `numpyro.distributions` instead of `scipy.stats`.

**Section 1** mirrors `chimera_pt_nuts.py` for an actual NUTS hyperlikelihood
run -- it needs the GWTC5 PE/injection files (cluster-only), so edit
`dir_data`/`dir_chain` before running it.

## The same necessary fix this notebook relies on

`feature_scales` used to be an `eqx.field(static=True)` tuple baked directly
into the fixed random draw (`W_hidden ~ N(0, 1/feature_scales[i]**2)` drawn
once). That makes it physically impossible to *sample*: a JAX `static` field
has to be identical, by value, across calls for the jit cache to reuse a
compiled trace, so NUTS proposing a new `feature_scales` at every leapfrog
step would force a fresh XLA compilation practically every step -- a
non-starter.

`CHIMERA/population/mass/paired/nn.py` now draws a fixed, feature_scales-
independent standard-normal noise tensor `raw_W_hidden` once, and applies the
lengthscale as an ordinary differentiable division at call time:
`W_hidden[k] = raw_W_hidden[k] / feature_scales[block_of(k)]`. This makes
`feature_scales` a genuine dynamic pytree leaf -- sampled, differentiated,
and `vmap`'d exactly like `w_out` already was, with a single jit trace
regardless of the concrete values drawn (and, for NUTS specifically, exactly
the differentiability NUTS needs to compute its gradient of the log density
w.r.t. `feature_scales` in the first place). See the class docstring in
`nn.py` for the full reasoning.

In [ ]:
import os, sys
os.environ["CHIMERA_ENABLE_GPU"] = "True"
sys.path.append(os.getcwd() + '/../')

import numpy as np
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import NUTS, MCMC
import matplotlib.pyplot as plt

from CHIMERA.population.mass.paired import random_features_density
from CHIMERA.population.mass import p_m1, p_m1m2

## Config

`default_feature_scales` are the 4 lengthscales from the class docstring
(broad-body -> narrow-peak). `scale_prior_width` sets how narrow the priors
around them are (fractional half-width in log-space); at 0.2 the 4 resulting
bands (`[0.8,1.2]`, `[0.24,0.36]`, `[0.08,0.12]`, `[0.024,0.036]`) are both
"quite narrow" and non-overlapping, which matters: if the bands overlapped,
the posterior would pick up a spurious label-switching degeneracy between
blocks (block *i* and block *i+1* trading places), since nothing else ties a
specific block to a specific physical role.

`num_warmup`/`num_samples`/`num_chains` are modest defaults for a *test*
run -- bump these up for a production fit, same spirit as
`chimera_pt_nuts.py`'s `n_warmup`/`n_samples` knobs (which use
parallel-tempering instead of a single chain; start from these smaller,
single-chain settings and move to `pt_nuts` if NUTS alone struggles with
multimodality).

In [ ]:
n_features = 64                                    # keep modest (tens, not hundreds) -- see nn.py docstring
default_feature_scales = (1.0, 0.3, 0.1, 0.03)     # the 4 lengthscale "blocks"
n_scales = len(default_feature_scales)
scale_prior_width = 0.2                            # +/-20% narrow, non-overlapping log-uniform bands
# Calibrate the w_out prior amplitude to n_features rather than hardcoding it:
# g(x) = sum_k w_out_k*cos(...), and since b_hidden ~ Uniform(0,2pi) independent of
# everything else, E[cos^2(.)] = 1/2 exactly regardless of feature_scales, so
# std(g) ~= w_out_prior_scale * sqrt(n_features/2). A too-wide w_out prior makes g (and
# hence p_tilde(m) = exp(g)/m) swing over many orders of magnitude, so the model
# concentrates its density in an essentially random sliver of mass space that rarely
# overlaps with any real event's PE samples -- every event then fails the pe_neff
# effective-sample-size check and the hyperlikelihood is -inf for every draw (see
# hyperlikelihood.compute_all in CHIMERA/likelihood.py). target_std_g=1-2 keeps g a mild,
# well-behaved modulation instead.
target_std_g = 1.2                                 # desired std of g(x) itself
w_out_prior_scale = target_std_g * (2.0 / n_features) ** 0.5

master_key = jax.random.PRNGKey(0)

# NUTS / MCMC run settings (Section 1) -- modest defaults for a *test* run;
# bump these up for a production fit.
num_warmup = 1000
num_samples = 1000
num_chains = 1
target_accept_prob = 0.8
max_tree_depth = 8
prng_seed = 0
chain_method = 'sequential'   # 'vectorized' can be faster on a single GPU if it fits in memory

dir_data = '/leonardo_work/IscrC_MLGW/gwtc5/data/'
dir_chain = '/leonardo_work/IscrC_MLGW/gwtc5/res/nuts/'
chain_label = 'gwtc5_randomfeatures_blocks_nuts'

## Priors

Same style as `chimera_pt_nuts.py`: `numpyro.distributions`-based
`uniform`/`loguniform`/`norm` helpers, one prior per hyperparameter.
`scale1`...`scale4` (one per lengthscale block) are sampled as individual
scalar sites (each block has different bounds); `w_out` is sampled as a
*single* `n_features`-length vector site sharing one `Normal` prior, via
`.expand([n_features]).to_event(1)` -- there is no `params_keys` flat-vector
bookkeeping to do, `numpyro` samples straight into a dict of named sites.

In [ ]:
def uniform(low, high):
  return dist.Uniform(low=low, high=high)

def loguniform(low, high):
  return dist.LogUniform(low=low, high=high)

def norm(loc, scale):
  return dist.Normal(loc=loc, scale=scale)

# --- mass-model (random_features_density) priors ---
mass_priors = {
  "beta": uniform(-4., 12.),               # mass-ratio pairing power law
  "m_low": uniform(0.4, 1.4),
  "m_high": uniform(50., 200.),
}

# narrow, non-overlapping priors on the 4 lengthscale blocks
scale_priors = {
  f"scale{i+1}": loguniform(s * (1. - scale_prior_width), s * (1. + scale_prior_width))
  for i, s in enumerate(default_feature_scales)
}

# single vector-valued Normal prior shared by every random-features output weight
w_out_prior = norm(loc=0., scale=w_out_prior_scale)

# --- cosmology / rate priors (same as chimera_pt_nuts.py) ---
cosmo_rate_priors = {
  "H0": uniform(10., 200.),
  "gamma": uniform(0., 12.),
  "kappa": uniform(0., 6.),
  "zp": uniform(0., 4.),
}

scalar_priors = {**cosmo_rate_priors, **mass_priors}  # every site that is a plain scalar
n_dim = len(scalar_priors) + n_scales + n_features
print(f"n_dim = {n_dim}  ({n_features} w_out + {n_scales} feature_scales + "
      f"{len(mass_priors)} mass + {len(cosmo_rate_priors)} cosmo/rate)")

## Section 0 -- local sanity checks (no cluster data / `numpyro`-cluster deps needed)

### 0.1 Prior-predictive draws across different lengthscale blocks

Draw `feature_scales` and `w_out` straight from the priors above (now
`numpyro.distributions.sample(key)` instead of `scipy.stats.rvs()`), build a
`random_features_density` realization for each draw, and overlay `p(m1)`.
Because the 4 scale priors are narrow, each draw is a small perturbation of
the default "blocks" -- this checks that the model behaves smoothly as
`feature_scales` moves around its prior, not that it produces wildly
different basis structures.

In [ ]:
def draw_scales_and_w_out(key):
  # Draw one (feature_scales, w_out) pair from the priors above.
  scale_key, w_key = jax.random.split(key)
  scale_keys = jax.random.split(scale_key, n_scales)
  fs = jnp.stack([scale_priors[f"scale{j+1}"].sample(scale_keys[j]) for j in range(n_scales)])
  w_out = w_out_prior.expand([n_features]).sample(w_key)
  return fs, w_out

n_draws = 12
m_low_fixed, m_high_fixed = 0.92, 124.2  # fixed mass range for this prior-predictive check

base_model = random_features_density(
  n_features=n_features, feature_scales=default_feature_scales,
  m_low=m_low_fixed, m_high=m_high_fixed, key=master_key,
)

rng_key = jax.random.PRNGKey(1)
m1 = jnp.linspace(m_low_fixed * 1.01, m_high_fixed * 0.99, 4000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
drawn_scales = []
for i in range(n_draws):
  rng_key, draw_key = jax.random.split(rng_key)
  fs, w_out = draw_scales_and_w_out(draw_key)
  drawn_scales.append(np.asarray(fs))

  model_i = base_model.update(feature_scales=fs, w_out=w_out)
  ax.plot(m1, p_m1(model_i, m1), alpha=0.7, lw=1.2)

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.set_title(f'{n_draws} prior-predictive draws of the 4 lengthscale blocks')
fig.tight_layout()
plt.show()

drawn_scales = np.array(drawn_scales)
print("drawn feature_scales -- min/max per block:")
for j in range(n_scales):
  print(f"  block {j+1} (default {default_feature_scales[j]:.3f}): "
        f"[{drawn_scales[:, j].min():.4f}, {drawn_scales[:, j].max():.4f}]")

### 0.2 Are the 4 blocks' priors actually narrow and non-overlapping?

Histogram many draws per block against the prior bounds.

In [ ]:
n_hist = 4000
fig, axes = plt.subplots(1, n_scales, figsize=(4 * n_scales, 3.2), sharey=False)
for j, ax in enumerate(axes):
  draws = np.asarray(scale_priors[f"scale{j+1}"].sample(jax.random.PRNGKey(100 + j), (n_hist,)))
  ax.hist(draws, bins=40, color=f"C{j}", alpha=0.8)
  ax.axvline(default_feature_scales[j], color='k', ls='--', lw=1, label='default')
  ax.set_title(f"block {j+1} prior")
  ax.set_xlabel("feature_scales[%d]" % j)
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

### 0.3 jit/vmap check: one compile, many `feature_scales` values

This is the actual point of the `nn.py` fix, and the property NUTS itself
depends on at every leapfrog step: batch several particles with *different*
`feature_scales` and `w_out` through `.update()` + `p_m1` under a single
`jax.jit` + `jax.vmap`, the same way `numpyro.infer.MCMC(..., chain_method='vectorized')`
would evaluate several chains at once, or the way a single NUTS trajectory
repeatedly re-evaluates the log density at new `feature_scales` proposals.
The second call reuses the trace (no recompilation) even though its
`feature_scales` values differ from the first call's.

In [ ]:
import time

n_particles_test = 64
batch_keys = jax.random.split(jax.random.PRNGKey(2), n_particles_test)
fs_batch, w_out_batch = jax.vmap(draw_scales_and_w_out)(batch_keys)

@jax.jit
def eval_particle(fs, w_out):
  model_i = base_model.update(feature_scales=fs, w_out=w_out)
  return p_m1(model_i, m1)

t0 = time.time()
out1 = jax.vmap(eval_particle)(fs_batch, w_out_batch)
out1.block_until_ready()
t1 = time.time()

fs_batch_2 = fs_batch * 1.05  # different concrete values, same shapes
out2 = jax.vmap(eval_particle)(fs_batch_2, w_out_batch)
out2.block_until_ready()
t2 = time.time()

print(f"first call (compile + run): {t1 - t0:.3f}s")
print(f"second call, different feature_scales values (should be much faster -- no retrace): {t2 - t1:.3f}s")
assert not jnp.any(jnp.isnan(out1)) and not jnp.any(jnp.isnan(out2))
print("no NaNs -- vmap over distinct feature_scales works with a single jit trace.")

## Section 1 -- full NUTS hyperlikelihood run (cluster only)

Mirrors `pt-nuts/examples/chimera_pt_nuts.py`'s structure (load GWTC5 PE +
injection data, build `cosmo`/`mass`/`rate` -> `population` ->
`hyperlikelihood`, define a `numpyro` model, run `numpyro.infer.MCMC(NUTS(...))`),
but as a single-temperature run rather than `chimera_pt_nuts.py`'s
parallel-tempered `pt_nuts.pt_nuts`, and with `random_features_density`
(sampled `feature_scales`) instead of the fixed-architecture `neural_density`.

Needs `dir_data`/`dir_chain` (set in the config cell above) to point at real
GWTC5 files -- **edit the paths before running this section**.

In [ ]:
from CHIMERA import data
from CHIMERA.cosmo import flrw
from CHIMERA.rate import madau_dickinson
from CHIMERA import population as population_module, selection_function, hyperlikelihood
import h5py

## Load PE data
file_ev = dir_data + 'GWTC5_242CBC_FAR0.25_PE2048.h5'
pe_gw = data.load_gw_pe_samples(file_ev, parameters=['m1det', 'm2det', 'dL'], return_struct=True)
pe_prior = jnp.load(dir_data + 'GWTC5_242CBC_FAR0.25_PE2048prior.npy')
pe_gw = pe_gw.update(pe_prior=pe_prior)
Nev = len(pe_gw.dL)

## Load injection data and define selection function
file_inj = dir_data + "GWTC5_injections_far0.25.h5"
f_inj = h5py.File(file_inj)
Ngen = float(f_inj.attrs['Ngen'])
Tobs = float(f_inj.attrs['Tobs'])
theta_inj_det = data.load_injection_data(file_inj, snr_cut=None, key_mapping={'snr': 'snr', 'log_pdraw': 'log_pdraw'}, frame='detector')

sel_fcn = selection_function(theta_inj_det, N_inj=Ngen)

In [ ]:
cosmo = flrw(H0=67.9, Om0=0.3065, z_max=5.)
mass = random_features_density(
  n_features=n_features, feature_scales=default_feature_scales,
  m_low=0.9, m_high=124., key=master_key,
)
rate = madau_dickinson()
population = population_module(cosmo, mass, rate, scale_free=True, Tobs=Tobs, R0=1)

hyperlike = hyperlikelihood(
  theta_gw_det=pe_gw,
  population=population,
  z_grids_res=300,
  selection_function=sel_fcn,
  pe_neff=2.0,
  inj_neff=None,
  kind_kde='fft',
  kernel='epan',
  kde_bw=None,
)

### The `numpyro` model

`chimera_model` is the generative counterpart of `random_features_blocks.ipynb`'s
`log_likelihood(params)`: one `numpyro.sample` call per scalar hyperparameter,
one per lengthscale block, and one vector-valued call for `w_out`, assembled
into the `hyperparams` dict that `hyperlikelihood.compute_all()` expects
(exactly the keys `random_features_density.update()`/`mass.keys` wants).
`hyperlike.__call__` already returns just the scalar log-hyperlikelihood
(`hyperlike.compute_log_hyperlike`), so a plain `numpyro.factor` is enough --
but `compute_all` is used instead so the per-event diagnostics (`log_like_evs`,
`N_exp`, `neff_inj`) can be attached as `numpyro.deterministic` sites. Doing
that turns them into ordinary posterior variables that `arviz.from_numpyro`
picks up automatically -- the `numpyro`/NUTS equivalent of `tempest`'s
`blobs`, with no separate bookkeeping needed.

In [ ]:
def chimera_model(hyperlike):
  hyperparams = {}
  for name, prior_dist in scalar_priors.items():
    hyperparams[name] = numpyro.sample(name, prior_dist)

  feature_scales = jnp.stack([
    numpyro.sample(f"scale{i+1}", scale_priors[f"scale{i+1}"]) for i in range(n_scales)
  ])
  hyperparams["feature_scales"] = feature_scales

  w_out = numpyro.sample("w_out", w_out_prior.expand([n_features]).to_event(1))
  hyperparams["w_out"] = w_out

  log_like_evs, N_exp, neff_inj, log_like = hyperlike.compute_all(**hyperparams)
  numpyro.deterministic("log_like_evs", log_like_evs)
  numpyro.deterministic("N_exp", N_exp)
  if neff_inj is not None:
    numpyro.deterministic("neff_inj", neff_inj)
  numpyro.factor("log_likelihood", log_like)

### Prior-predictive finite-likelihood check

Before launching NUTS, check how much of the prior mass gives a finite
log-likelihood -- NUTS still needs to be *initialized* somewhere finite (by
default `numpyro` retries random re-initialization internally, but a prior
that is mostly `-inf` wastes that budget and warmup). Unlike the
`scipy.stats`-based Python loop in `random_features_blocks.ipynb`, priors
here are `numpyro.distributions`, so the whole check is `jit`+`vmap`'d over
`n_check` prior draws in one shot -- the same one-compile-many-particles
pattern as the 0.3 check above, just applied to the full hyperparameter set
including `hyperlike.compute_all` itself.

In [ ]:
def draw_prior(key):
  # Draw one full hyperparams dict (cosmo/rate/mass + scales + w_out) from the priors.
  n_scalar = len(scalar_priors)
  keys = jax.random.split(key, n_scalar + n_scales + 1)
  hyperparams = {name: d.sample(keys[i]) for i, (name, d) in enumerate(scalar_priors.items())}
  feature_scales = jnp.stack([
    scale_priors[f"scale{j+1}"].sample(keys[n_scalar + j]) for j in range(n_scales)
  ])
  hyperparams["feature_scales"] = feature_scales
  hyperparams["w_out"] = w_out_prior.expand([n_features]).sample(keys[-1])
  return hyperparams

@jax.jit
def draw_and_eval(key):
  hyperparams = draw_prior(key)
  ll = hyperlike(**hyperparams)
  return ll, hyperparams

n_check = 10000
check_keys = jax.random.split(jax.random.PRNGKey(3), n_check)
ll_batch, hyperparams_batch = jax.vmap(draw_and_eval)(check_keys)
finite_mask = jnp.isfinite(ll_batch)
n_finite = int(jnp.sum(finite_mask))
print(f"{n_finite}/{n_check} prior draws gave a finite log-likelihood "
      f"({n_finite/n_check*100:.2f}% of the prior)")

### What does the finite-likelihood region of the prior look like as a mass function?

Before spending compute on the full run, plot `p(m1)` for a handful of the
finite-likelihood draws collected above -- these are draws that passed every
event's `pe_neff` check, so this is a sanity check on what NUTS's *starting*
region actually looks like, not yet the posterior.

In [ ]:
finite_idx = np.asarray(jnp.where(finite_mask)[0])
n_plot = min(20, len(finite_idx))
plot_idx = np.random.default_rng(0).choice(finite_idx, size=n_plot, replace=False)
m1_grid = jnp.linspace(0.5, 200., 3000)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
for i in plot_idx:
  mass_i = mass.update(
    w_out=hyperparams_batch["w_out"][i], feature_scales=hyperparams_batch["feature_scales"][i],
    beta=hyperparams_batch["beta"][i], m_low=hyperparams_batch["m_low"][i], m_high=hyperparams_batch["m_high"][i],
  )
  ax.plot(m1_grid, p_m1(mass_i, m1_grid), alpha=0.6, lw=1)

ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("$m_1$"); ax.set_ylabel("$p(m_1)$")
ax.set_title(f"{n_plot} finite-likelihood prior draws")
fig.tight_layout()
plt.show()

### Run NUTS

`chain_method='sequential'` runs `num_chains` chains one after another on a
single device (safe default for a single GPU); switch to `'vectorized'` to
`vmap` them together if they fit in memory at once, or `'parallel'` across
multiple local devices. `target_accept_prob`/`max_tree_depth` are the same
knobs `chimera_pt_nuts.py` forwards into `pt_nuts` (there called
`target_acceptance_rate`/`max_num_doublings`).

In [ ]:
nuts_kernel = NUTS(chimera_model, target_accept_prob=target_accept_prob, max_tree_depth=max_tree_depth)
mcmc = MCMC(
  nuts_kernel,
  num_warmup=num_warmup,
  num_samples=num_samples,
  num_chains=num_chains,
  chain_method=chain_method,
  progress_bar=True,
)

mcmc.run(jax.random.PRNGKey(prng_seed), hyperlike=hyperlike)
mcmc.print_summary(exclude_deterministic=True)

### Save to `arviz.InferenceData`

`arviz.from_numpyro(mcmc)` reads `mcmc.get_samples(group_by_chain=True)`
directly -- both the sampled hyperparameters (`H0`, ..., `scale1`...`scale4`,
`w_out`) and the `numpyro.deterministic` blobs (`log_like_evs`, `N_exp`,
`neff_inj`) land in `idata.posterior`, and divergences/tree-depth/step-size
land in `idata.sample_stats`, with no manual `xarray.Dataset` assembly
needed (contrast with the `tempest` notebook's Section 1, which builds
`posterior_ds`/`sample_stats_ds` by hand from `blobs`).

In [ ]:
import arviz as az

idata = az.from_numpyro(mcmc)
n_divergent = int(idata.sample_stats.diverging.sum())
print(f"divergent transitions: {n_divergent} / {num_chains * num_samples}")

idata.to_netcdf(f"{dir_chain}/{chain_label}_final.nc")

## Section 2 -- inspecting the 4 blocks in the posterior

Compares the posterior over the 4 `feature_scales` against their (narrow)
priors -- if the posterior tightens noticeably relative to the prior for a
block, the data is informative about that lengthscale; if it just traces the
prior, that block isn't constrained by this dataset/`n_features` and could
be dropped or widened. Also draws a posterior-predictive `p(m1)` band using
the sampled `w_out`/`feature_scales`/mass-edge hyperparameters together. Both
cells below flatten `idata.posterior`'s `(chain, draw)` dims into one axis of
i.i.d. posterior draws -- unlike the `tempest` notebook, there is no
`weights` array to resample against first.

In [ ]:
fig, axes = plt.subplots(1, n_scales, figsize=(4 * n_scales, 3.2))
for j, ax in enumerate(axes):
  post = idata.posterior[f"scale{j+1}"].values.reshape(-1)
  prior_draws = np.asarray(scale_priors[f"scale{j+1}"].sample(jax.random.PRNGKey(200 + j), (5000,)))
  ax.hist(prior_draws, bins=40, density=True, alpha=0.4, label='prior', color='gray')
  ax.hist(post, bins=40, density=True, alpha=0.7, label='posterior', color=f"C{j}")
  ax.axvline(default_feature_scales[j], color='k', ls='--', lw=1)
  ax.set_title(f"block {j+1}")
  ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

In [ ]:
n_ppc = 200
w_out_post = idata.posterior["w_out"].values.reshape(-1, n_features)
scales_post = np.stack([idata.posterior[f"scale{j+1}"].values.reshape(-1) for j in range(n_scales)], axis=-1)
beta_post = idata.posterior["beta"].values.reshape(-1)
m_low_post = idata.posterior["m_low"].values.reshape(-1)
m_high_post = idata.posterior["m_high"].values.reshape(-1)

idx = np.random.default_rng(0).choice(len(w_out_post), size=n_ppc, replace=False)
m1_grid = jnp.linspace(0.5, 150., 3000)

curves = []
for i in idx:
  model_i = mass.update(
    w_out=w_out_post[i], feature_scales=scales_post[i],
    beta=beta_post[i], m_low=m_low_post[i], m_high=m_high_post[i],
  )
  curves.append(p_m1(model_i, m1_grid))

curves = jnp.stack(curves)
lo, med, hi = jnp.percentile(curves, jnp.array([5., 50., 95.]), axis=0)

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.fill_between(m1_grid, lo, hi, alpha=0.3, label='90% posterior-predictive')
ax.plot(m1_grid, med, lw=1.5, label='median')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('$m_1$'); ax.set_ylabel('$p(m_1)$')
ax.legend()
fig.tight_layout()
plt.show()